In [1]:
import os
import sys
import re
import numpy as np
import pandas as pd
import plotly.express as px
import scipy.stats as stats
from dash import html, dcc, Input, Output, Dash
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sys.path.append(os.path.join(os.getcwd(), '..', '..'))
from baseVR.base_functionality import init_import_paths
init_import_paths() 

from CustomLogger import CustomLogger as Logger
from analytics_processing import analytics

from analytics_processing.sessions_from_nas_parsing import sessionlist_fullfnames_from_args, fullfnames2snames
from dashsrc.plot_components.plots import plot_TrackFiringRate
from dashsrc.plot_components.plots import plot_unit_fr_stability


In [2]:
Logger().init_logger(None, None, logging_level="DEBUG")
animal_ids = [6]
paradigm = [1100]
session_range = [1,33]
session_ids = None
normalize = True
smooth = False
excl_session_names = None  #['2024-11-29_17-21_rYL006_P1100_LinearTrackStop_28min', '2025-01-21_18-49_rYL006_P1100_LinearTrackStop_30min', '2024-12-11_17-42_rYL006_P1100_LinearTrackStop_30min']

session_dirs = sessionlist_fullfnames_from_args(paradigm, animal_ids, session_ids, excl_session_names=excl_session_names)[0]
session_names= fullfnames2snames(session_dirs)

2026-02-03 12:21:37,223|DEBUG|86282|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Searching NAS for applicable sessions...
2026-02-03 12:21:38,947|DEBUG|86282|sessions_from_nas_parsing|get_sessionlist_fullfnames
	For paradigms [1100], animals [6], found 34 sessions.
2026-02-03 12:21:39,050|DEBUG|86282|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: [1100], animal_ids: [6], session_ids: None, from_date: None, to_date: None
	Merging 34 sessions



In [3]:
# firing rates and behavior data
fr = analytics.get_analytics('FiringRate40msHz', session_names=session_names)
fr_track = analytics.get_analytics('FiringRateTrackwiseHz', session_names=session_names)
fr_z_scored  = analytics.get_analytics('FiringRate40msZ', session_names=session_names)
fr_z_all_sess = fr.apply(lambda unit_fr: ((unit_fr - unit_fr.mean()) / unit_fr.std()))
behav = analytics.get_analytics('BehaviorTrackwise', session_names=session_names)
behav.index = behav.index.droplevel(('animal_id', 'paradigm_id', 'entry_id'))

2026-02-03 12:21:39,196|DEBUG|86282|sessions_from_nas_parsing|sessionnames2fullfnames
	Inferring NAS paths for list of session names...
2026-02-03 12:21:39,658|DEBUG|86282|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: None, animal_ids: None, session_ids: None, from_date: None, to_date: None
	Merging 34 sessions

2026-02-03 12:21:39,676|DEBUG|86282|analytics|get_analytics
	Processing FiringRate40msHz, (1100, 6, '2024-11-14_15-01') 2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min.hdf5
/Volumes/large/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min
2026-02-03 12:21:39,704|INFO|86282|analytics|get_analytics
	Analytic `FiringRate40msHz` does not exist for (1100, 6, '2024-11-14_15-01'), compute first, or check for typo
2026-02-03 12:21:39,706|DEBUG|86282|analytics|get_analytics
	Processing FiringRate40msHz, (1100, 6, '2024-11-14_16-40') 2024-11-14_16-40_rYL006_P1100_LinearTrackStop_21min.hdf5

In [ ]:
# ensamble related data
ensambles = analytics.get_analytics('ConcatenatedEnsambles40ms', session_names=session_names)
ens_data = analytics.get_analytics('TrackwiseEnsembleProj', session_names=session_names)
ensamble_proj = analytics.get_analytics('ConcatenatedEnsambleProj40ms', session_names=session_names)#.drop("to_ephys_timestamp", axis=1)
ensamble_proj.set_index(['session_id'], inplace = True)
ens_data = ens_data.set_index(['session_id', 'trial_id']).sort_index()

2026-02-03 12:23:18,012|DEBUG|86282|sessions_from_nas_parsing|sessionnames2fullfnames
	Inferring NAS paths for list of session names...
2026-02-03 12:23:18,363|DEBUG|86282|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: None, animal_ids: None, session_ids: None, from_date: None, to_date: None
	Merging 34 sessions

2026-02-03 12:23:18,364|DEBUG|86282|analytics|get_analytics
	Processing ConcatenatedEnsambles40ms, for n=34 sessions
2026-02-03 12:23:18,408|INFO|86282|analytics|get_analytics
	Loaded animal-level analytic `ConcatenatedEnsambles40ms` last modified on 2025-07-23T14:15:05.148205
2026-02-03 12:23:18,408|DEBUG|86282|sessions_from_nas_parsing|sessionnames2fullfnames
	Inferring NAS paths for list of session names...
2026-02-03 12:23:18,411|DEBUG|86282|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: None, animal_ids: None, session_ids: None, from_date: None, to_date: None
	Merging 34 sessions

2026-02-03 12:23:18,411|DEBUG|86282|an

/Volumes/large/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min/../../animal_analytics/ConcatenatedEnsambles40ms.parquet
/Volumes/large/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min/../../animal_analytics/TrackwiseEnsembleProj.parquet


2026-02-03 12:23:35,525|INFO|86282|analytics|get_analytics
	Loaded animal-level analytic `TrackwiseEnsembleProj` last modified on 2026-01-30T17:48:01.007341
2026-02-03 12:23:35,555|DEBUG|86282|sessions_from_nas_parsing|sessionnames2fullfnames
	Inferring NAS paths for list of session names...
2026-02-03 12:23:35,602|DEBUG|86282|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: None, animal_ids: None, session_ids: None, from_date: None, to_date: None
	Merging 34 sessions

2026-02-03 12:23:35,604|DEBUG|86282|analytics|get_analytics
	Processing ConcatenatedEnsambleProj40ms, for n=34 sessions


/Volumes/large/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min/../../animal_analytics/ConcatenatedEnsambleProj40ms.parquet


2026-02-03 12:24:14,899|INFO|86282|analytics|get_analytics
	Loaded animal-level analytic `ConcatenatedEnsambleProj40ms` last modified on 2025-07-23T14:16:02.733331


In [5]:
# t0 bin
t0_events = analytics.get_analytics('TrialWiseT0Events40ms', session_names=session_names)
t0_events

2026-02-03 12:24:15,728|DEBUG|86282|sessions_from_nas_parsing|sessionnames2fullfnames
	Inferring NAS paths for list of session names...
2026-02-03 12:24:16,028|DEBUG|86282|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: None, animal_ids: None, session_ids: None, from_date: None, to_date: None
	Merging 34 sessions

2026-02-03 12:24:16,030|DEBUG|86282|analytics|get_analytics
	Processing TrialWiseT0Events40ms, (1100, 6, '2024-11-14_15-01') 2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min.hdf5
/Volumes/large/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min
2026-02-03 12:24:16,081|INFO|86282|analytics|get_analytics
	Analytic `TrialWiseT0Events40ms` does not exist for (1100, 6, '2024-11-14_15-01'), compute first, or check for typo
2026-02-03 12:24:16,082|DEBUG|86282|analytics|get_analytics
	Processing TrialWiseT0Events40ms, (1100, 6, '2024-11-14_16-40') 2024-11-14_16-40_rYL006_P1100_LinearTrack

trial_id  cue  trial_outcome  \
paradigm_id animal_id session_id       entry_id                                 
1100        6         2024-11-14_16-40 0                1  1.0            0.0   
                                       1                1  1.0            0.0   
                                       2                1  1.0            0.0   
                                       3                1  1.0            0.0   
                                       4                1  1.0            0.0   
...                                                   ...  ...            ...   
                      2025-01-27_13-39 755            152  2.0            1.0   
                                       756            152  2.0            1.0   
                                       757            152  2.0            1.0   
                                       758            152  2.0            1.0   
                                       759            152  2.0            1.0   

                                                 choice_R1  choice_R2  \
paradigm_id animal_id session_id       entry_id                         
1100        6         2024-11-14_16-40 0             False      False   
                                       1             False      False   
                                       2             False      False   
                                       3             False      False   
                                       4             False      False   
...                                                    ...        ...   
                      2025-01-27_13-39 755           False       True   
                                       756           False       True   
                                       757           False       True   
                                       758           False       True   
                                       759           False       True   

                                                     t0_event_name  \
paradigm_id animal_id session_id       entry_id                      
1100        6         2024-11-14_16-40 0           cueZone_visible   
                                       1             cueZone_entry   
                                       2              cueZone_exit   
                                       3         enter_reward1Zone   
                                       4         enter_reward2Zone   
...                                                            ...   
                      2025-01-27_13-39 755         cueZone_visible   
                                       756           cueZone_entry   
                                       757            cueZone_exit   
                                       758       enter_reward1Zone   
                                       759       enter_reward2Zone   

                                                           t0  x_position  \
paradigm_id animal_id session_id       entry_id                             
1100        6         2024-11-14_16-40 0         4.600000e+06 -119.732063   
                                       1         5.400000e+06  -79.536247   
                                       2         6.760000e+06   24.898020   
                                       3         7.080000e+06   51.438919   
                                       4         9.480000e+06  170.376450   
...                                                       ...         ...   
                      2025-01-27_13-39 755       4.393240e+09 -120.656754   
                                       756       4.394000e+09  -79.428299   
                                       757       4.396000e+09   25.843262   
                                       758       4.396480e+09   49.770477   
                                       759       4.398880e+09  170.561493   

                                                 x_alignment  \
paradigm_id animal_id session_id       entry_id                
1100

In [8]:
len(t0_events.index.get_level_values('session_id').unique())

for session_id in t0_events.index.get_level_values('session_id').unique():
    

29

In [ ]:
df = ens_data

# If session_id is in the index, bring it back as a column
if "session_id" not in df.columns:
    if isinstance(df.index, pd.MultiIndex) and "session_id" in df.index.names:
        df = df.reset_index()
    elif df.index.name == "session_id":
        df = df.reset_index()

trial_avg = (
    df.groupby(["session_id", "from_position_bin", "cue"], as_index=False)["Assembly012"]
      .mean()
      .rename(columns={"Assembly012": "Assembly012_mean"})
)

trial_avg = trial_avg[trial_avg["cue"].isin([1, 2])].copy()

# sorting x-axis values
pos_numeric = pd.to_numeric(trial_avg["from_position_bin"], errors="coerce")
if pos_numeric.notna().any():
    trial_avg["_pos"] = pos_numeric
    x_order = np.sort(trial_avg["_pos"].dropna().unique())
    # keep a numeric x for correct ordering/spacing
    trial_avg["_x"] = trial_avg["_pos"]
    x_ticks = x_order
    x_ticktext = [str(int(x)) if float(x).is_integer() else str(x) for x in x_order]
else:
    # fallback: treat as ordered categorical by appearance
    trial_avg["_x"] = trial_avg["from_position_bin"].astype(str)
    x_order = list(pd.unique(trial_avg["_x"]))
    x_ticks = x_order
    x_ticktext = x_order

# Ridgeline plot parameters
sessions = sorted(trial_avg["session_id"].unique())
dy = 0.8          # vertical spacing between sessions
cue_offset = 0.02 # vertical separation between cue 1 and cue 2 within a session
amp = 0.2

fig = go.Figure()

for i, s in enumerate(sessions):
    base = i * dy

    for cue, off in [(1, -cue_offset), (2, +cue_offset)]:
        sub = trial_avg[(trial_avg["session_id"] == s) & (trial_avg["cue"] == cue)].copy()

        # Ensure every x bin exists, fill missing with 0 (or np.nan if you prefer gaps)
        if pos_numeric.notna().any():
            sub = sub.set_index("_x").reindex(x_order).reset_index()
        else:
            sub = sub.set_index("_x").reindex(x_order).reset_index()

        y = sub["Assembly012_mean"].fillna(0.0).to_numpy()
        y_ridge = base + off + amp * y

        fig.add_trace(
            go.Scatter(
                x=sub["_x"],
                y=y_ridge,
                mode="lines",
                line=dict(width=1, color= cue == 1 and "orange" or "purple"),
                fill= None,
                name=f"session {s} | cue {cue}",
                hovertemplate=(
                    "session=%{customdata[0]}<br>"
                    "cue=%{customdata[1]}<br>"
                    "from_position_bin=%{x}<br>"
                    "mean(Assembly012)=%{customdata[2]:.4f}<extra></extra>"
                ),
                customdata=np.c_[np.full(len(sub), s), np.full(len(sub), cue), y],
                showlegend=True,
            )
        )



#Layout polish
fig.update_layout(
    title="Ridgeline: mean(Assembly012) per from_position_bin, one ridge per session",
    xaxis_title="from_position_bin",
    yaxis_title="session_id (stacked ridges)",
    # hovermode="x",
    height=max(450, 40 * len(sessions) + 200),
)

# Put session labels on the y-axis at each session baseline
fig.update_yaxes(
    tickmode="array",
    tickvals=[i * dy for i in range(len(sessions))],
    ticktext=[str(s) for s in sessions],
)

fig.add_vrect(x0=-80, x1=25,  fillcolor="orange", opacity=0.2, layer="below", line_width=0)
fig.add_vrect(x0=50,  x1=110, fillcolor="grey",   opacity=0.2, layer="below", line_width=0)
fig.add_vrect(x0=170, x1=230, fillcolor="grey",   opacity=0.2, layer="below", line_width=0)


if pos_numeric.notna().any():
    fig.update_xaxes(
        tickmode="array",
        ticktext=x_ticktext,
        showgrid=False, zeroline=False
    )

fig.show()

In [27]:
ens = ensamble_proj.copy()
t0 = t0_events.copy()

# Ensure columns exist (index -> columns if needed)
if isinstance(ens.index, pd.MultiIndex):
    ens = ens.reset_index()
elif ens.index.name is not None:
    ens = ens.reset_index()

if isinstance(t0.index, pd.MultiIndex):
    t0 = t0.reset_index()
elif t0.index.name is not None:
    t0 = t0.reset_index()

# Choose the assembly to plot
ens_col = "Assembly012"

# Midpoint time for each 40 ms bin
ens["t_mid"] = 0.5 * (ens["from_ephys_timestamp"].to_numpy() + ens["to_ephys_timestamp"].to_numpy())

import numpy as np
import pandas as pd

ens = ensamble_proj.copy()
t0  = t0_events.copy()

# Ensure columns exist
if isinstance(ens.index, pd.MultiIndex) or ens.index.name is not None:
    ens = ens.reset_index()
if isinstance(t0.index, pd.MultiIndex) or t0.index.name is not None:
    t0 = t0.reset_index()

# Midpoint time of each bin
ens["t_mid"] = 0.5 * (ens["from_ephys_timestamp"].to_numpy() + ens["to_ephys_timestamp"].to_numpy())

# 1) Build per-trial table from t0_events
# Identify interval columns (your analytic uses many "*_interval" columns)
interval_cols = [c for c in t0.columns if c.endswith("_interval")]

# Condense to one row per (session_id, trial_id) and compute trial_interval from ALL intervals
def build_trial_interval(g: pd.DataFrame) -> pd.Interval | None:
    lefts, rights = [], []
    for col in interval_cols:
        vals = g[col].dropna()
        if vals.empty:
            continue
        # vals should be pandas.Interval objects
        for iv in vals:
            lefts.append(iv.left)
            rights.append(iv.right)
    if len(lefts) == 0:
        return None
    return pd.Interval(min(lefts), max(rights), closed="both")

# Keep stable per-trial metadata (cue/t0 can be taken as first non-null)
meta_cols = [c for c in ["cue", "t0"] if c in t0.columns]
zone_cols = [c for c in ["cue_entry_interval", "R1_entry_interval", "R2_entry_interval"] if c in t0.columns]

t0_trial = (
    t0.groupby(["session_id", "trial_id"], as_index=False)
      .first()[["session_id", "trial_id"] + meta_cols + zone_cols]
)

trial_iv = (
    t0.groupby(["session_id", "trial_id"], as_index=False)
      .apply(build_trial_interval)
      .reset_index(name="trial_interval")
)

t0_trial = t0_trial.merge(trial_iv, on=["session_id", "trial_id"], how="left")

# Drop trials without any interval information
t0_trial = t0_trial.dropna(subset=["trial_interval"]).copy()

# 2) Assign each ensemble bin to a trial_id within each session
assigned = []
debug_rows = []

for s, ens_s in ens.groupby("session_id", sort=False):
    trials_s = t0_trial[t0_trial["session_id"] == s].copy()

    if trials_s.empty:
        debug_rows.append((s, "no_trials", np.nan, np.nan, np.nan, np.nan))
        continue

    iv = pd.IntervalIndex(trials_s["trial_interval"].tolist())

    t_mid = ens_s["t_mid"].to_numpy()
    idx = iv.get_indexer(t_mid)  # -1 if not contained

    n_ok = int((idx >= 0).sum())

    # Diagnostics: compare ranges
    ens_min, ens_max = float(np.nanmin(t_mid)), float(np.nanmax(t_mid))
    tri_min = float(min([x.left for x in iv]))
    tri_max = float(max([x.right for x in iv]))
    debug_rows.append((s, "ok" if n_ok > 0 else "no_match", n_ok, ens_min, ens_max, tri_min, tri_max))

    if n_ok == 0:
        continue

    ens_s = ens_s.copy()
    ens_s = ens_s.loc[idx >= 0].copy()
    ens_s["trial_id"] = trials_s["trial_id"].to_numpy()[idx[idx >= 0]]
    assigned.append(ens_s)

# If still empty, print a compact debug table to see what is wrong
if len(assigned) == 0:
    dbg = pd.DataFrame(
        debug_rows,
        columns=["session_id", "status", "n_assigned", "ens_t_min", "ens_t_max", "trial_t_min", "trial_t_max"]
    ).sort_values("session_id")
    display(dbg)
    raise ValueError("No bins could be assigned to trials. Inspect dbg above (time-base mismatch or trial intervals do not cover ensemble timestamps).")

ens_assigned = pd.concat(assigned, ignore_index=True)

# -------------------------
# 3) Ridgeline signal: mean activation per (session, cue, aligned time)
# -------------------------
ridge = (
    ens_assigned.groupby(["session_id", "cue", "t_bin"], as_index=False)[ens_col]
    .mean()
    .rename(columns={ens_col: "ens_mean"})
)

# -------------------------
# 4) Zone occupancy fractions per (session, cue, aligned time)
# -------------------------
def _in_interval(iv, x):
    if iv is None or (isinstance(iv, float) and np.isnan(iv)):
        return False
    return (x in iv)

ens_assigned["in_cue"] = ens_assigned.apply(lambda r: _in_interval(r["cue_entry_interval"], r["t_mid"]), axis=1)
ens_assigned["in_r1"]  = ens_assigned.apply(lambda r: _in_interval(r["R1_entry_interval"],  r["t_mid"]), axis=1)
ens_assigned["in_r2"]  = ens_assigned.apply(lambda r: _in_interval(r["R2_entry_interval"],  r["t_mid"]), axis=1)

occ = (
    ens_assigned.groupby(["session_id", "cue", "t_bin"], as_index=False)[["in_cue", "in_r1", "in_r2"]]
    .mean()
    .rename(columns={"in_cue": "f_cue", "in_r1": "f_r1", "in_r2": "f_r2"})
)

plot_df = ridge.merge(occ, on=["session_id", "cue", "t_bin"], how="left").fillna(0.0)

# -------------------------
# 5) Plot: ridgeline with time-varying saturation via short segments
# -------------------------
def hex_to_rgb01(h):
    h = h.lstrip("#")
    return np.array([int(h[i:i+2], 16) for i in (0, 2, 4)], dtype=float) / 255.0

def rgb01_to_hex(rgb01):
    rgb = np.clip(np.round(rgb01 * 255.0), 0, 255).astype(int)
    return "#{:02x}{:02x}{:02x}".format(rgb[0], rgb[1], rgb[2])

def mix_zone_color(f_cue, f_r1, f_r2,
                   cue_hex="#ff7f0e", r1_hex="#2ca02c", r2_hex="#9467bd",
                   background_hex="#ffffff"):
    cue = hex_to_rgb01(cue_hex)
    r1  = hex_to_rgb01(r1_hex)
    r2  = hex_to_rgb01(r2_hex)
    bg  = hex_to_rgb01(background_hex)

    f = np.array([f_cue, f_r1, f_r2], dtype=float)
    f = np.clip(f, 0.0, 1.0)
    s = f.sum()
    if s > 1.0:
        f = f / s
        s = 1.0

    rgb = bg * (1.0 - s) + cue * f[0] + r1 * f[1] + r2 * f[2]
    return rgb01_to_hex(rgb)

sessions = sorted(plot_df["session_id"].unique())
dy = 0.8
cue_offset = 0.03
amp = 0.25
downsample = 1  # raise to 2..5 if too slow

fig = go.Figure()

for i, s in enumerate(sessions):
    base = i * dy
    for cue, off in [(1, -cue_offset), (2, +cue_offset)]:
        sub = plot_df[(plot_df["session_id"] == s) & (plot_df["cue"] == cue)].sort_values("t_bin")
        if sub.empty:
            continue
        sub = sub.iloc[::downsample].reset_index(drop=True)

        x = sub["t_bin"].to_numpy()
        y = sub["ens_mean"].to_numpy()
        y_r = base + off + amp * y

        fc = sub["f_cue"].to_numpy()
        f1 = sub["f_r1"].to_numpy()
        f2 = sub["f_r2"].to_numpy()

        for k in range(len(sub) - 1):
            c = mix_zone_color(fc[k], f1[k], f2[k])
            fig.add_trace(
                go.Scatter(
                    x=[x[k], x[k+1]],
                    y=[y_r[k], y_r[k+1]],
                    mode="lines",
                    line=dict(width=1.6, color=c),
                    name=f"session {s} | cue {cue}",
                    showlegend=(k == 0),
                    customdata=[[s, cue, x[k], y[k], fc[k], f1[k], f2[k]]],
                    hovertemplate=(
                        "session=%{customdata[0]}<br>"
                        "cue=%{customdata[1]}<br>"
                        "t_rel_s=%{customdata[2]:.3f}<br>"
                        f"mean({ens_col})=%{{customdata[3]:.4f}}<br>"
                        "f_cue=%{customdata[4]:.2f}, f_R1=%{customdata[5]:.2f}, f_R2=%{customdata[6]:.2f}"
                        "<extra></extra>"
                    ),
                )
            )

fig.update_layout(
    title=f"Ridgeline over aligned time: mean({ens_col}) from ensamble_proj (colored by zone occupancy fraction)",
    xaxis_title="Aligned time (s) relative to t0",
    yaxis_title="session_id (stacked ridges)",
    height=max(450, 40 * len(sessions) + 200),
)
fig.update_yaxes(
    tickmode="array",
    tickvals=[i * dy for i in range(len(sessions))],
    ticktext=[str(s) for s in sessions],
)
fig.update_xaxes(showgrid=False, zeroline=False)

fig.show()


/var/folders/ml/fyq5x5t55wx4129dn03n32hm0000gn/T/ipykernel_86282/2104634433.py:66: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



TypeError: DataFrame.reset_index() got an unexpected keyword argument 'name'

In [30]:
fr_track

from_position_bin  trial_id  \
paradigm_id animal_id session_id       entry_id                                
1100        6         2024-11-14_16-40 0                    -169.0         1   
                                       1                    -169.0         2   
                                       2                    -169.0         3   
                                       3                    -169.0         4   
                                       4                    -169.0         5   
...                                                            ...       ...   
                      2025-01-27_13-39 65488                 262.0       123   
                                       65489                 262.0       124   
                                       65490                 262.0       128   
                                       65491                 262.0       135   
                                       65492                 262.0       139   

                                                 Unit0001   Unit0002  \
paradigm_id animal_id session_id       entry_id                        
1100        6         2024-11-14_16-40 0         1.666667   0.000000   
                                       1         0.000000   0.000000   
                                       2         0.000000   3.333333   
                                       3         3.333333   0.000000   
                                       4         1.666667   0.000000   
...                                                   ...        ...   
                      2025-01-27_13-39 65488     0.000000   0.000000   
                                       65489     0.000000   0.000000   
                                       65490     0.000000  12.500000   
                                       65491     0.000000   0.000000   
                                       65492     0.000000   0.000000   

                                                 Unit0003  Unit0004  Unit0005  \
paradigm_id animal_id session_id       entry_id                                 
1100        6         2024-11-14_16-40 0              0.0       0.0  1.666667   
                                       1              0.0       0.0  0.000000   
                                       2              0.0       0.0  0.000000   
                                       3              0.0       0.0  0.000000   
                                       4              0.0       0.0  1.666667   
...                                                   ...       ...       ...   
                      2025-01-27_13-39 65488          0.0       0.0  0.000000   
                                       65489          0.0       0.0  0.000000   
                                       65490          0.0      12.5  0.000000   
                                       65491          0.0       0.0  0.000000   
                                       65492          0.0       0.0  0.000000   

                                                 Unit0006  Unit0007  Unit0008  \
paradigm_id animal_id session_id       entry_id                                 
1100        6         2024-11-14_16-40 0              0.0       0.0       0.0   
                                       1              0.0       0.0       0.0   
                                       2              0.0       0.0       0.0   
                                       3              0.0       0.0       0.0   
                                       4              0.0       0.0       0.0   
...                                                   ...       ...       ...   
                      2025-01-27_13-39 65488          0.0       0.0       0.0   
                                       65489          0.0      25.0       0.0   
                                       65490          0.0       0.0       0.0   
                                       65491         12.5      25.0       0.0   
                                       65492 